# Задание 1

#### Для начала импортируем необходимые для предобработки и выполнения задания библиотеки

In [1]:
import pandas as pd
import numpy as np

#### Далее используем pandas для чтения данных

In [2]:
#Аэропорты («Airport database») – названия колонок начинаются с Airport ID до Source:
airport_url = 'https://raw.githubusercontent.com/jpatokal/openflights/master/data/airports.dat'
#Маршрутная сеть («Route database») – названия колонок начинаются с Airline до Equipment:
route_url = 'https://raw.githubusercontent.com/jpatokal/openflights/master/data/routes.dat'

In [3]:
airport_datebase = pd.read_csv(airport_url, header = None, names = ['Airport_ID','Name','City', 'Country', 'IATA', 'ICAO', 
    'Latitude', 'Longitude', 'Altitude', 'Timezone', 'DST', 
    'Tz database time zone', 'Type', 'Source']) # название колонок из сайта, указанного в задании
airport_datebase.head()

,Airport_ID,Name,City,Country,IATA,ICAO,Latitude,Longitude,Altitude,Timezone,DST,Tz database time zone,Type,Source
0,1,Goroka Airport,Goroka,Papua New Guinea,GKA,AYGA,-6.081690,145.391998,5282,10,U,Pacific/Port_Moresby,airport,OurAirports
1,2,Madang Airport,Madang,Papua New Guinea,MAG,AYMD,-5.207080,145.789001,20,10,U,Pacific/Port_Moresby,airport,OurAirports
2,3,Mount Hagen Kagamuga Airport,Mount Hagen,Papua New Guinea,HGU,AYMH,-5.826790,144.296005,5388,10,U,Pacific/Port_Moresby,airport,OurAirports
3,4,Nadzab Airport,Nadzab,Papua New Guinea,LAE,AYNZ,-6.569803,146.725977,239,10,U,Pacific/Port_Moresby,airport,OurAirports
4,5,Port Moresby Jacksons International Airport,Port Moresby,Papua New Guinea,POM,AYPY,-9.443380,147.220001,146,10,U,Pacific/Port_Moresby,airport,OurAirports


In [4]:
routes_cols = [
    'Airline', 'Airline ID', 'Source Airport', 'Source Airport ID',
    'Destination Airport', 'Destination Airport ID', 'Codeshare', 
    'Stops', 'Equipment'
] # название колонок из сайта, указанного в задании
routes = pd.read_csv(route_url, header=None, names=routes_cols)
routes.head()

,Airline,Airline ID,Source Airport,Source Airport ID,Destination Airport,Destination Airport ID,Codeshare,Stops,Equipment
0,2B,410,AER,2965,KZN,2990,NaN,0,CR2
1,2B,410,ASF,2966,KZN,2990,NaN,0,CR2
2,2B,410,ASF,2966,MRV,2962,NaN,0,CR2
3,2B,410,CEK,2968,KZN,2990,NaN,0,CR2
4,2B,410,CEK,2968,OVB,4078,NaN,0,CR2


#### Далее сделаем, чтобы у всех ID был тип 'str', это упростит выполнение задания

In [5]:
airport_datebase['Airport_ID'] = airport_datebase['Airport_ID'].astype(str)
routes['Source Airport ID'] = routes['Source Airport ID'].astype(str)
routes['Destination Airport ID'] = routes['Destination Airport ID'].astype(str)

#### Находим для каждого аэропорта количество взлетов

In [6]:
departures = routes['Source Airport ID'].value_counts()
departures

Source Airport ID
3682    915
3830    558
3364    535
507     527
1382    524
       ... 
2517      1
2014      1
5884      1
2018      1
6334      1
Name: count, Length: 3321, dtype: int64

#### Находим для каждого аэропорта количество посадок

In [7]:
arrivals = routes['Destination Airport ID'].value_counts()
arrivals

Destination Airport ID
3682    911
3830    550
3364    534
507     524
1382    517
       ... 
9828      1
908       1
5809      1
7054      1
6792      1
Name: count, Length: 3327, dtype: int64

#### Далее соединяем полученные данные в один датафрейм
#### Переименовываем столбцы для удобства
#### Заполняем пропущенные значения нулями

In [8]:
route_count = pd.concat([departures, arrivals], axis=1)
route_count = route_count.fillna(0)
route_count.columns = ['Взлеты', 'Посадки']
route_count

,Взлеты,Посадки
3682,915.0,911.0
3830,558.0,550.0
3364,535.0,534.0
507,527.0,524.0
1382,524.0,517.0
...,...,...
6372,0.0,1.0
7369,0.0,1.0
7176,0.0,1.0
9758,0.0,1.0


#### Теперь можно легко посчитать сумму взлетов и посадок

In [9]:
route_count['Сумма взлет/посадок'] = route_count['Взлеты']+route_count['Посадки']
route_count

,Взлеты,Посадки,Сумма взлет/посадок
3682,915.0,911.0,1826.0
3830,558.0,550.0,1108.0
3364,535.0,534.0,1069.0
507,527.0,524.0,1051.0
1382,524.0,517.0,1041.0
...,...,...,...
6372,0.0,1.0,1.0
7369,0.0,1.0,1.0
7176,0.0,1.0,1.0
9758,0.0,1.0,1.0


#### Чтобы Airport ID были указаны в отдельно столбце, а не в индексах, проделываем следующее:

In [10]:
route_count.reset_index(inplace=True)
route_count = route_count.rename(columns={'index':'Airport_ID'})
route_count

,Airport_ID,Взлеты,Посадки,Сумма взлет/посадок
0,3682,915.0,911.0,1826.0
1,3830,558.0,550.0,1108.0
2,3364,535.0,534.0,1069.0
3,507,527.0,524.0,1051.0
4,1382,524.0,517.0,1041.0
...,...,...,...,...
3329,6372,0.0,1.0,1.0
3330,7369,0.0,1.0,1.0
3331,7176,0.0,1.0,1.0
3332,9758,0.0,1.0,1.0


#### Остается только соединить 2 датафрейма (airport_datebase и route_count)
#### У начальной базы данных при использовании merge оставляю только нужные столбцы

In [11]:
route_airport_merge = route_count.merge(airport_datebase[['Airport_ID', 'Name', 'Country']], on='Airport_ID', how = 'left')
route_airport_merge = route_airport_merge[['Country', 'Name', 'Взлеты', 'Посадки', 'Сумма взлет/посадок']]
route_airport_merge

,Country,Name,Взлеты,Посадки,Сумма взлет/посадок
0,United States,Hartsfield Jackson Atlanta International Airport,915.0,911.0,1826.0
1,United States,Chicago O'Hare International Airport,558.0,550.0,1108.0
2,China,Beijing Capital International Airport,535.0,534.0,1069.0
3,United Kingdom,London Heathrow Airport,527.0,524.0,1051.0
4,France,Charles de Gaulle International Airport,524.0,517.0,1041.0
...,...,...,...,...,...
3329,Mongolia,Dalanzadgad Airport,0.0,1.0,1.0
3330,Brazil,Santana do Araguaia Airport,0.0,1.0,1.0
3331,NaN,NaN,0.0,1.0,1.0
3332,NaN,NaN,0.0,1.0,1.0


#### Сортирую значения и вывожу топ 10
#### Устанавливаю индексы, как требуется в задании

In [12]:
route_airport_merge = route_airport_merge.sort_values('Сумма взлет/посадок', ascending = False)
route_airport_merge.set_index(['Country', 'Name']).head(10)

Взлеты  \
Country        Name                                                       
United States  Hartsfield Jackson Atlanta International Airport   915.0   
               Chicago O'Hare International Airport               558.0   
China          Beijing Capital International Airport              535.0   
United Kingdom London Heathrow Airport                            527.0   
France         Charles de Gaulle International Airport            524.0   
Germany        Frankfurt am Main Airport                          497.0   
United States  Los Angeles International Airport                  492.0   
               Dallas Fort Worth International Airport            469.0   
               John F Kennedy International Airport               456.0   
Netherlands    Amsterdam Airport Schiphol                         453.0   

                                                                 Посадки  \
Country        Name                                                        
United States  Hartsfield Jackson Atlanta International Airport    911.0   
               Chicago O'Hare International Airport                550.0   
China          Beijing Capital International Airport               534.0   
United Kingdom London Heathrow Airport                             524.0   
France         Charles de Gaulle International Airport             517.0   
Germany        Frankfurt am Main Airport                           493.0   
United States  Los Angeles International Airport                   498.0   
               Dallas Fort Worth International Airport             467.0   
               John F Kennedy International Airport                455.0   
Netherlands    Amsterdam Airport Schiphol                          450.0   

                                                                 Сумма взлет/посадок  
Country        Name                                                                   
United States  Hartsfield Jackson Atlanta International Airport               1826.0  
               Chicago O'Hare International Airport                           1108.0  
China          Beijing Capital International Airport                          1069.0  
United Kingdom London Heathrow Airport                                        1051.0  
France         Charles de Gaulle International Airport                        1041.0  
Germany        Frankfurt am Main Airport                                       990.0  
United States  Los Angeles International Airport                               990.0  
               Dallas Fort Worth International Airport                         936.0  
               John F Kennedy International Airport                            911.0  
Netherlands    Amsterdam Airport Schiphol                                      903.0

# Задание 2

### Пропущенные значения

#### Выведем все столбцы, где есть хотя бы одно пропущенное значение (NA)

In [13]:
# Для базы данных с аэропортами
# Исследовав столбцы, рассмотрев их уникальные значения, понимаю, что NaN записаны ещё как \N, \\N

columns_with_miss_airport = airport_datebase.columns[
    (airport_datebase == r'\N').any() | 
    (airport_datebase == '\\N').any() |
    airport_datebase.isna().any()
].tolist()
columns_with_miss_airport

['City', 'IATA', 'ICAO', 'Timezone', 'DST', 'Tz database time zone']

In [14]:
# Для базы данных с маршрутами аналогично

columns_with_miss_routes = routes.columns[
    (routes == r'\N').any() | 
    (routes == '\\N').any() |
    routes.isna().any()
].tolist()
columns_with_miss_routes

['Airline ID',
 'Source Airport ID',
 'Destination Airport ID',
 'Codeshare',
 'Equipment']

Построила цикл, который сможет помочь, найти ошибочные данные

In [15]:
for column in airport_datebase.columns:
    unique_values = airport_datebase[column].describe()
    print(f"Описание столбца '{column}':")
    print(unique_values)

Описание столбца 'Airport_ID':
count     7698
unique    7698
top          1
freq         1
Name: Airport_ID, dtype: object
Описание столбца 'Name':
count                  7698
unique                 7658
top       San Pedro Airport
freq                      3
Name: Name, dtype: object
Описание столбца 'City':
count       7649
unique      6955
top       London
freq           9
Name: City, dtype: object
Описание столбца 'Country':
count              7698
unique              237
top       United States
freq               1512
Name: Country, dtype: object
Описание столбца 'IATA':
count     7698
unique    6073
top         \N
freq      1626
Name: IATA, dtype: object
Описание столбца 'ICAO':
count     7698
unique    7698
top       AYGA
freq         1
Name: ICAO, dtype: object
Описание столбца 'Latitude':
count    7698.000000
mean       25.808442
std        28.404946
min       -90.000000
25%         6.907983
50%        34.085701
75%        47.239551
max        89.500000
Name: Latitude, dtype: 

Выявила, что в данных минимальная высота над уровнем моря -1266, а в реальной жизни, она не может быть ниже -430,5\
Выведем строки с некорректной высотой

### Некорректная высота над уровнем моря

In [16]:
invalid_alt = airport_datebase[airport_datebase['Altitude']<-430.5]
invalid_alt

,Airport_ID,Name,City,Country,IATA,ICAO,Latitude,Longitude,Altitude,Timezone,DST,Tz database time zone,Type,Source
1558,1600,Bar Yehuda Airfield,Metzada,Israel,MTZ,LLMZ,31.328199,35.388599,-1266,2,E,Asia/Jerusalem,airport,OurAirports


### Неверно записанные ID аэропортов 
То есть не все Source Airport ID и Destination Airport ID из routes существуют. Это значит, что их нет в Airport_ID из airport_datebase.

In [17]:
invalid_source_ids = routes[~routes['Source Airport ID'].isin(airport_datebase['Airport_ID'])]['Source Airport ID'].unique().tolist()
invalid_dest_ids = routes[~routes['Destination Airport ID'].isin(airport_datebase['Airport_ID'])]['Destination Airport ID'].unique().tolist()
invalid_source_ids # неверно записанные ID аэропортов для взлетов

['\\N',
 '7167',
 '7168',
 '7164',
 '7173',
 '7144',
 '7145',
 '7147',
 '5475',
 '8858',
 '7215',
 '7119',
 '7110',
 '7112',
 '9741',
 '7113',
 '4385',
 '8202',
 '9742',
 '7212',
 '7216',
 '7218',
 '8201',
 '7099',
 '8203',
 '8492',
 '7238',
 '7219',
 '6721',
 '7239',
 '7241',
 '7200',
 '7193',
 '8204',
 '5955',
 '8206',
 '8207',
 '5957',
 '7901',
 '7895',
 '5958',
 '7163',
 '7165',
 '9747',
 '5804',
 '7118',
 '5454',
 '7029',
 '7287',
 '7288',
 '5455',
 '6463',
 '7356',
 '7358',
 '5477',
 '9823',
 '5712',
 '7310',
 '6428',
 '5979',
 '5980',
 '5984',
 '6476',
 '7528',
 '9841',
 '6750',
 '5601',
 '9827',
 '6132',
 '4248',
 '9745',
 '7155',
 '7153',
 '7102',
 '6449',
 '6448',
 '7478',
 '7479',
 '6919',
 '7138',
 '7137',
 '2611',
 '6231',
 '9181',
 '8735',
 '7237',
 '7150',
 '7151',
 '7152',
 '5805',
 '9540',
 '8174',
 '6457',
 '8170',
 '8173',
 '5855',
 '6770',
 '9757',
 '9756',
 '10119',
 '8962',
 '9842',
 '7562',
 '9135',
 '4389',
 '7141',
 '7243',
 '4264',
 '6129',
 '6135']

In [18]:
invalid_dest_ids # неверно записанные ID аэропортов для посадок

['\\N',
 '7167',
 '7168',
 '7164',
 '7163',
 '7176',
 '7173',
 '7165',
 '7172',
 '7145',
 '7144',
 '7147',
 '8858',
 '5475',
 '7215',
 '7119',
 '7112',
 '9741',
 '7113',
 '7110',
 '7218',
 '4385',
 '8202',
 '7099',
 '7212',
 '9742',
 '7216',
 '8201',
 '8203',
 '8492',
 '7219',
 '7200',
 '7238',
 '7239',
 '7193',
 '6721',
 '7241',
 '8207',
 '5955',
 '5957',
 '7901',
 '7895',
 '5958',
 '9758',
 '8204',
 '8206',
 '9747',
 '5804',
 '7118',
 '5454',
 '6463',
 '7287',
 '7288',
 '5455',
 '7029',
 '7356',
 '5477',
 '7358',
 '9823',
 '5712',
 '7310',
 '6428',
 '5980',
 '5984',
 '5979',
 '6476',
 '7528',
 '9841',
 '6750',
 '9827',
 '5601',
 '6132',
 '4248',
 '9745',
 '7102',
 '7155',
 '7153',
 '6448',
 '6449',
 '7478',
 '7479',
 '6919',
 '7137',
 '7138',
 '6231',
 '9181',
 '8735',
 '7152',
 '7151',
 '7237',
 '7150',
 '5805',
 '8170',
 '8173',
 '6457',
 '8174',
 '9540',
 '6770',
 '5855',
 '9757',
 '9756',
 '2611',
 '10119',
 '8962',
 '9842',
 '7562',
 '9135',
 '4389',
 '7141',
 '7243',
 '4264',
 

### ICAO коды, которые записаны не 4 буквами, а 3 

In [19]:
not_four = airport_datebase[(airport_datebase['ICAO'].str.len()<4)|(airport_datebase['ICAO'].str.len()>4)]['ICAO'].tolist()
not_four

['Q51', '03N', '5A8', 'KQA', 'S31', '\\N']

### Несколько ID для одного и того же аэропорта

In [70]:
duplicate_airports = airport_datebase[airport_datebase.\
duplicated(subset=['Name'], keep=False)].\
groupby('Name')['Airport_ID'].apply(list)

In [71]:
duplicate_airports

Name
Arlington Municipal Airport                 [7846, 11141]
Bathurst Airport                             [5538, 6249]
Capital City Airport                   [3544, 8284, 8631]
Charlottetown Airport                        [182, 13341]
Columbus Municipal Airport                 [11816, 11847]
Deer Lake Airport                              [45, 5532]
Douglas Municipal Airport                    [7588, 8273]
El Dorado Airport                            [2481, 2837]
Faro Airport                                  [205, 1626]
Flinders Island Airport                     [6267, 11038]
Flores Airport                              [1625, 13735]
Henderson Field                              [2253, 9942]
La Florida Airport                           [2662, 2716]
Las Brujas Airport                          [2717, 10171]
Lawrence Municipal Airport                   [8288, 8366]
Linden Airport                               [2872, 8550]
Mae Hong Son Airport                         [4086, 4159]
Melbourne

### Ошибки, найденные во время визуализации

#### Неправильно записанные города и страны

Из визуализации я увидела аэропорты, у которых неправильно опеределены города и страны. С помощью построений и визуализаций я выяснила IATA и, если неизвестен IATA код, я определяла ICAO код, для того, чтобы в дальнейшем вручную подредактировать страны и города.

In [20]:
airport_datebase['Country'] = np.where(airport_datebase['IATA'] == 'HLE', 'Saint Helena' , airport_datebase.Country )
airport_datebase['City'] = np.where(airport_datebase['IATA'] == 'HLE' , 'Jamestown' , airport_datebase.City)
airport_datebase['Country'] = np.where(airport_datebase['IATA'] == 'EMP', 'United States' , airport_datebase.Country )
airport_datebase['City'] = np.where(airport_datebase['IATA'] == 'EMP' , 'Emporia' , airport_datebase.City)
airport_datebase['Country'] = np.where(airport_datebase['IATA'] == 'LND', 'United States' , airport_datebase.Country)
airport_datebase['City'] = np.where(airport_datebase['IATA'] == 'LND' , 'Lander' , airport_datebase.City)
airport_datebase['City'] = np.where(airport_datebase['IATA'] == 'OCN' , ' San Diego County' , airport_datebase.City)
airport_datebase['Country'] = np.where(airport_datebase['IATA'] == 'OCN' , 'United States' , airport_datebase.Country)
airport_datebase['Country'] = np.where(airport_datebase['IATA'] == 'OEL' , 'Russia' , airport_datebase.Country)
airport_datebase['City'] = np.where(airport_datebase['IATA'] == 'OEL' , 'Orel' , airport_datebase.City)
airport_datebase['Country'] = np.where(airport_datebase['IATA'] == 'TZR' , 'Hungary' , airport_datebase.Country)
airport_datebase['City'] = np.where(airport_datebase['IATA'] == 'TZR' , 'Taszar' , airport_datebase.City)
airport_datebase['Country'] = np.where(airport_datebase['ICAO'] == 'MRGT' , 'United States' , airport_datebase.Country)
airport_datebase['City'] = np.where(airport_datebase['ICAO'] == 'MRGT' , 'Florida' , airport_datebase.City)
airport_datebase['Country'] = np.where(airport_datebase['ICAO'] == 'KNDZ' , 'United States' , airport_datebase.Country)
airport_datebase['City'] = np.where(airport_datebase['ICAO'] == 'KNDZ' , 'Florida' , airport_datebase.City)
airport_datebase['Country'] = np.where(airport_datebase['ICAO'] == 'FA08' , 'United States' , airport_datebase.Country)
airport_datebase['City'] = np.where(airport_datebase['ICAO'] == 'FA08' , 'Florida' , airport_datebase.City)
airport_datebase['Country'] = np.where(airport_datebase['ICAO'] == 'SNKV' , 'Brazil' , airport_datebase.Country)
airport_datebase['Country']=np.where(airport_datebase['ICAO']=='KW63', 'United States', airport_datebase.Country)
airport_datebase['Country']=np.where(airport_datebase['ICAO']=='WA98', 'United States', airport_datebase.Country)
airport_datebase['Country']=np.where(airport_datebase['ICAO']=='WA19', 'United States', airport_datebase.Country)
airport_datebase['Country']=np.where(airport_datebase['ICAO']=='SCRT', 'Chile', airport_datebase.Country)
airport_datebase['Country']=np.where(airport_datebase['ICAO']=='SLCR', 'Bolivia', airport_datebase.Country)
airport_datebase['Country'] = np.where(airport_datebase['ICAO'] == 'SVFT' , 'Venezuela' , airport_datebase.Country)
airport_datebase['Country'] = np.where(airport_datebase['ICAO'] == 'SVSJ' , 'Venezuela' , airport_datebase.Country)
airport_datebase['City'] = np.where(airport_datebase['IATA'] == 'OCN' , 'California' , airport_datebase.City)
airport_datebase['Country'] = np.where(airport_datebase['IATA'] == 'OCN' , 'United States' , airport_datebase.Country)

#### Неверные координаты

In [21]:
# У аэропорта с ICAO кодом CDG2 неправильно определена долгота, он должен быть во Франции
airport_datebase['Longitude'] = np.where(airport_datebase['ICAO']=='CDG2', 2.32, airport_datebase.Longitude)

# Задание 3

In [22]:
#pip install pycountry pycountry-convert

#### Для присоединения континентов буду использовать вспомогательные библиотеки

In [23]:
import pycountry
import pycountry_convert as pc

#### Функция для определения континента по названию страны

In [24]:
def get_continent(country_name):
    try:
        country_code = pc.country_name_to_country_alpha2(country_name) #Преобразуем название страны в двухбуквенный код
        continent_code = pc.country_alpha2_to_continent_code(country_code) #По коду страны получаем код континента
        continent_name = pc.convert_continent_code_to_continent_name(continent_code) #Преобразуем код континента в полное название
        return continent_name
    #Если на каком-то этапе произошла ошибка (например, страна не найдена), возвращаем NaN
    except Exception:
        return np.nan
#Применяем функцию
airport_datebase['Continent'] = airport_datebase['Country'].apply(get_continent)

In [25]:
airport_datebase['Continent'].unique() # Видим, что есть nan

array(['Oceania', 'North America', 'Europe', 'Africa', nan,
       'South America', 'Asia'], dtype=object)

#### Обработаем NaN значения в столбце Continent

In [26]:
airport_datebase[airport_datebase['Continent'].isna()]

,Airport_ID,Name,City,Country,IATA,ICAO,Latitude,Longitude,Altitude,Timezone,DST,Tz database time zone,Type,Source,Continent
250,253,Port Bouet Airport,Abidjan,Cote d'Ivoire,ABJ,DIAP,5.261390,-3.926290,21,0,N,Africa/Abidjan,airport,OurAirports,NaN
251,254,Bouaké Airport,Bouake,Cote d'Ivoire,BYK,DIBK,7.738800,-5.073670,1230,0,N,Africa/Abidjan,airport,OurAirports,NaN
252,255,Daloa Airport,Daloa,Cote d'Ivoire,DJO,DIDL,6.792810,-6.473190,823,0,N,Africa/Abidjan,airport,OurAirports,NaN
253,256,Korhogo Airport,Korhogo,Cote d'Ivoire,HGO,DIKO,9.387180,-5.556660,1214,0,N,Africa/Abidjan,airport,OurAirports,NaN
254,257,Man Airport,Man,Cote d'Ivoire,MJC,DIMN,7.272070,-7.587360,1089,0,N,Africa/Abidjan,airport,OurAirports,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7240,12003,Meiktila Air Base,Meiktila,Burma,\N,VYML,20.886400,95.892792,699,\N,\N,\N,airport,OurAirports,NaN
7400,13180,St. Helena Airport,Jamestown,Saint Helena,HLE,FHSH,-15.957725,-5.645943,1017,0,U,\N,airport,OurAirports,NaN
7410,13212,Odienne Airport,Odienne,Cote d'Ivoire,KEO,DIOD,9.500000,-7.567000,1365,0,U,\N,airport,OurAirports,NaN
7421,13260,Oyo Ollombo Airport,Oyo,Congo (Brazzaville),OLL,FCOD,-1.226666,15.910000,1073,1,N,\N,airport,OurAirports,NaN


In [27]:
airport_datebase[airport_datebase.Continent.isna()]['Country'].unique()

array(["Cote d'Ivoire", 'Congo (Brazzaville)', 'Congo (Kinshasa)',
       'Saint Helena', 'Reunion', 'Antarctica', 'West Bank',
       'Midway Islands', 'Virgin Islands', 'Netherlands Antilles',
       'Burma', 'East Timor', 'Johnston Atoll', 'Western Sahara',
       'Wake Island'], dtype=object)

#### Вручную определяю для данных стран континент

In [28]:
airport_datebase['Continent'] = np.where(airport_datebase['Country'] == "Cote d'Ivoire" , 'Africa' , airport_datebase.Continent)
airport_datebase['Continent'] = np.where(airport_datebase['Country'] == "Congo (Brazzaville)" , 'Africa' , airport_datebase.Continent)
airport_datebase['Continent'] = np.where(airport_datebase['Country'] == "Congo (Kinshasa)" , 'Africa' , airport_datebase.Continent)
airport_datebase['Continent'] = np.where(airport_datebase['Country'] == "Saint Helena" , 'Africa' , airport_datebase.Continent)
airport_datebase['Continent'] = np.where(airport_datebase['Country'] == "Reunion" , 'Africa' , airport_datebase.Continent)
airport_datebase['Continent'] = np.where(airport_datebase['Country'] == "Antarctica" , 'Antarctica' , airport_datebase.Continent)
airport_datebase['Continent'] = np.where(airport_datebase['Country'] == "West Bank" , 'Asia' , airport_datebase.Continent)
airport_datebase['Continent'] = np.where(airport_datebase['Country'] == "Netherlands Antilles" , 'South America' , airport_datebase.Continent)
airport_datebase['Continent'] = np.where(airport_datebase['Country'] == "East Timor" , 'Asia' , airport_datebase.Continent)
airport_datebase['Continent'] = np.where(airport_datebase['Country'] == "Western Sahara" , 'Africa' , airport_datebase.Continent)
airport_datebase['Continent'] = np.where(airport_datebase['Country'] == "Midway Islands" , 'North America' , airport_datebase.Continent)
airport_datebase['Continent'] = np.where(airport_datebase['Country'] == "Virgin Islands" , 'North America' , airport_datebase.Continent)
airport_datebase['Continent'] = np.where(airport_datebase['Country'] == "Burma" , 'Asia' , airport_datebase.Continent)
airport_datebase['Continent'] = np.where(airport_datebase['Country'] == "Johnston Atoll" , 'North America' , airport_datebase.Continent)
airport_datebase['Continent'] = np.where(airport_datebase['Country'] == "Wake Island" , 'Oceania' , airport_datebase.Continent)

In [29]:
airport_datebase[airport_datebase['Continent'].isna()]

,Airport_ID,Name,City,Country,IATA,ICAO,Latitude,Longitude,Altitude,Timezone,DST,Tz database time zone,Type,Source,Continent


Всё, больше пропущенных значений в столбце Continent нет

#### Позже уже при визуализации я поняла, что для всех городов России континент определяется "Европа", поэтому я решила именно часть данных с Россией поделить по континентам с помощью Уральских гор

In [30]:
# Маска для России
is_russia = airport_datebase["Country"] == "Russia"

# Обновляем "Continent" только для российских аэропортов
airport_datebase.loc[is_russia, "Continent"] = np.where(
    airport_datebase.loc[is_russia, "Longitude"] < 60, "Europe", "Asia")

# Для Чукотки необходимо определять вручную, потому что это особенности системы отсчёта
airport_datebase['Continent'] = np.where(airport_datebase['ICAO'] == 'UHML' , 'Asia' , airport_datebase.Continent)
airport_datebase['Continent'] = np.where(airport_datebase['ICAO'] == 'UHMI' , 'Asia' , airport_datebase.Continent)
airport_datebase['Continent'] = np.where(airport_datebase['IATA'] == 'PVS' , 'Asia' , airport_datebase.Continent)


In [31]:
airport_datebase.head()

,Airport_ID,Name,City,Country,IATA,ICAO,Latitude,Longitude,Altitude,Timezone,DST,Tz database time zone,Type,Source,Continent
0,1,Goroka Airport,Goroka,Papua New Guinea,GKA,AYGA,-6.081690,145.391998,5282,10,U,Pacific/Port_Moresby,airport,OurAirports,Oceania
1,2,Madang Airport,Madang,Papua New Guinea,MAG,AYMD,-5.207080,145.789001,20,10,U,Pacific/Port_Moresby,airport,OurAirports,Oceania
2,3,Mount Hagen Kagamuga Airport,Mount Hagen,Papua New Guinea,HGU,AYMH,-5.826790,144.296005,5388,10,U,Pacific/Port_Moresby,airport,OurAirports,Oceania
3,4,Nadzab Airport,Nadzab,Papua New Guinea,LAE,AYNZ,-6.569803,146.725977,239,10,U,Pacific/Port_Moresby,airport,OurAirports,Oceania
4,5,Port Moresby Jacksons International Airport,Port Moresby,Papua New Guinea,POM,AYPY,-9.443380,147.220001,146,10,U,Pacific/Port_Moresby,airport,OurAirports,Oceania


# Задание 3.1

Почитав документацию Bokeh, я решила, что проще будет использовать другую библиотеку, так как она более интерактивная. Поискала разные пакеты, наткнулась на Folium, решила использовать её

In [32]:
#pip install folium

In [33]:
import folium

/Users/anastasiafirsova/repo/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


#### Начнем с Европы

In [34]:
europa = airport_datebase[(airport_datebase['Continent']=='Europe')]

m_eur = folium.Map()

#Через цикл добавляем все точки

lat = europa['Latitude']
lon = europa['Longitude']
iata = europa['IATA']

# Аэропорт с кодом IATA "SBH" принадлежит Франции, поэтому определяется, как Европа
# Аэропорт с кодом EGAR принадлежит Великобритании web: https://en.m.wikipedia.org/wiki/Rothera_Air_Facility

for lat, lon, iata in zip(lat , lon, iata) :
    folium.Marker(location=[lat, lon], popup='IATA: '+str(iata), icon=folium.Icon(color = 'green')).add_to(m_eur)
m_eur  

#### Азия

In [35]:
asia = airport_datebase[(airport_datebase['Continent']=='Asia')]

m_asia = folium.Map()

#Через цикл добавляем все точки

lat = asia['Latitude']
lon = asia['Longitude']
iata = asia['IATA']

# Аэропорт с ICAO кодом WA19 принадлежит Индонезии, поэтому определяется как Азия

for lat, lon, iata in zip(lat , lon, iata) :
    folium.Marker(location=[lat, lon], popup='IATA: '+str(iata), icon=folium.Icon(color = 'green')).add_to(m_asia)
m_asia  

#### Океания

In [36]:
oceania = airport_datebase[(airport_datebase['Continent']=='Oceania')]

m_oceania = folium.Map()

#Через цикл добавляем все точки

lat = oceania['Latitude']
lon = oceania['Longitude']
iata = oceania['IATA']

for lat, lon, iata in zip(lat , lon, iata) :
    folium.Marker(location=[lat, lon], popup='IATA: '+str(iata), icon=folium.Icon(color = 'green')).add_to(m_oceania)
m_oceania 

#### Северная Америка

In [37]:
north_america = airport_datebase[(airport_datebase['Continent']=='North America')]

m_north_america = folium.Map()

#Через цикл добавляем все точки

lat = north_america['Latitude']
lon = north_america['Longitude']
iata = north_america['IATA']

for lat, lon, iata in zip(lat , lon, iata) :
    folium.Marker(location=[lat, lon], popup='IATA: '+str(iata), icon=folium.Icon(color = 'green')).add_to(m_north_america)
m_north_america

#### Южная Америка

In [38]:
south_america = airport_datebase[(airport_datebase['Continent']=='South America')]

m_south_america = folium.Map()

#Через цикл добавляем все точки

lat = south_america['Latitude']
lon = south_america['Longitude']
iata = south_america['IATA']

for lat, lon, iata in zip(lat , lon, iata) :
    folium.Marker(location=[lat, lon], popup='IATA: '+str(iata), icon=folium.Icon(color = 'green')).add_to(m_south_america)
m_south_america

#### Антаркида

In [39]:
antarctica = airport_datebase[(airport_datebase['Continent']=='Antarctica')]

m_antarctica = folium.Map()

#Через цикл добавляем все точки

lat = antarctica['Latitude']
lon = antarctica['Longitude']
iata = antarctica['IATA']

for lat, lon, iata in zip(lat , lon, iata) :
    folium.Marker(location=[lat, lon], popup='IATA: '+str(iata), icon=folium.Icon(color = 'green')).add_to(m_antarctica)
m_antarctica

# Задание 3.2

#### В airport_datebase уже имеются данные о континенте, осталось посчитать количество взлетов и посадок, а потом сгруппировать по континенту.

In [40]:
# Сначала работаю с routes, с помощью него посчитаю количество взлетов и посадок, как в первом задании.
departures = routes['Source Airport ID'].value_counts().reset_index()\
.rename(columns = {'count':'Arrivals', 'Source Airport ID':'Airport_ID'})
arrivals = routes['Destination Airport ID'].value_counts().reset_index()\
.rename(columns = {'count':'Destination', 'Destination Airport ID':'Airport_ID'})
route_count = departures.merge(arrivals, on = 'Airport_ID')
route_count = route_count.fillna(0)
route_count['Сумма взлет/посадок'] = route_count['Destination']+route_count['Arrivals']
route_count_merge = route_count.merge(airport_datebase, on = 'Airport_ID') #присоединяем к данным информацию об аэропортах

In [41]:
route_count

,Airport_ID,Arrivals,Destination,Сумма взлет/посадок
0,3682,915,911,1826
1,3830,558,550,1108
2,3364,535,534,1069
3,507,527,524,1051
4,1382,524,517,1041
...,...,...,...,...
3309,2455,1,1,2
3310,2517,1,1,2
3311,2014,1,1,2
3312,2018,1,1,2


In [42]:
route_count_merge.head()

,Airport_ID,Arrivals,Destination,Сумма взлет/посадок,Name,City,Country,IATA,ICAO,Latitude,Longitude,Altitude,Timezone,DST,Tz database time zone,Type,Source,Continent
0,3682,915,911,1826,Hartsfield Jackson Atlanta International Airport,Atlanta,United States,ATL,KATL,33.636700,-84.428101,1026,-5,A,America/New_York,airport,OurAirports,North America
1,3830,558,550,1108,Chicago O'Hare International Airport,Chicago,United States,ORD,KORD,41.978600,-87.904800,672,-6,A,America/Chicago,airport,OurAirports,North America
2,3364,535,534,1069,Beijing Capital International Airport,Beijing,China,PEK,ZBAA,40.080101,116.584999,116,8,U,Asia/Shanghai,airport,OurAirports,Asia
3,507,527,524,1051,London Heathrow Airport,London,United Kingdom,LHR,EGLL,51.470600,-0.461941,83,0,E,Europe/London,airport,OurAirports,Europe
4,1382,524,517,1041,Charles de Gaulle International Airport,Paris,France,CDG,LFPG,49.012798,2.550000,392,1,E,Europe/Paris,airport,OurAirports,Europe


In [43]:
route_count_merge.sort_values(by = 'Сумма взлет/посадок', ascending = False)

,Airport_ID,Arrivals,Destination,Сумма взлет/посадок,Name,City,Country,IATA,ICAO,Latitude,Longitude,Altitude,Timezone,DST,Tz database time zone,Type,Source,Continent
0,3682,915,911,1826,Hartsfield Jackson Atlanta International Airport,Atlanta,United States,ATL,KATL,33.636700,-84.428101,1026,-5,A,America/New_York,airport,OurAirports,North America
1,3830,558,550,1108,Chicago O'Hare International Airport,Chicago,United States,ORD,KORD,41.978600,-87.904800,672,-6,A,America/Chicago,airport,OurAirports,North America
2,3364,535,534,1069,Beijing Capital International Airport,Beijing,China,PEK,ZBAA,40.080101,116.584999,116,8,U,Asia/Shanghai,airport,OurAirports,Asia
3,507,527,524,1051,London Heathrow Airport,London,United Kingdom,LHR,EGLL,51.470600,-0.461941,83,0,E,Europe/London,airport,OurAirports,Europe
4,1382,524,517,1041,Charles de Gaulle International Airport,Paris,France,CDG,LFPG,49.012798,2.550000,392,1,E,Europe/Paris,airport,OurAirports,Europe
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2815,2860,1,1,2,Santa Elena de Uairen Airport,Santa Ana De Uairen,Venezuela,SNV,SVSE,4.555000,-61.150002,2938,-4,U,America/Caracas,airport,OurAirports,South America
2816,2864,1,1,2,San Tomé Airport,San Tome,Venezuela,SOM,SVST,8.945147,-64.151085,861,-4,U,America/Caracas,airport,OurAirports,South America
2817,1983,1,1,2,Takapoto Airport,Takapoto,French Polynesia,TKP,NTGT,-14.709500,-145.246002,12,-10,U,Pacific/Tahiti,airport,OurAirports,Oceania
2818,206,1,1,2,Fort Mcpherson Airport,Fort Mcpherson,Canada,ZFM,CZFM,67.407501,-134.860992,116,-7,A,America/Edmonton,airport,OurAirports,North America


#### Для каждого континента по отдельности найдем крупнейшие аэропорты, а потом соединим всё в один датафрейм 
#### ЕВРОПА

In [44]:
eur_largest = route_count_merge[route_count_merge['Continent']=='Europe']
eur_largest.sort_values(by = 'Сумма взлет/посадок', ascending = False)
eur_largest = eur_largest.head(5)
eur_largest

,Airport_ID,Arrivals,Destination,Сумма взлет/посадок,Name,City,Country,IATA,ICAO,Latitude,Longitude,Altitude,Timezone,DST,Tz database time zone,Type,Source,Continent
3,507,527,524,1051,London Heathrow Airport,London,United Kingdom,LHR,EGLL,51.470600,-0.461941,83,0,E,Europe/London,airport,OurAirports,Europe
4,1382,524,517,1041,Charles de Gaulle International Airport,Paris,France,CDG,LFPG,49.012798,2.550000,392,1,E,Europe/Paris,airport,OurAirports,Europe
5,340,497,493,990,Frankfurt am Main Airport,Frankfurt,Germany,FRA,EDDF,50.033333,8.570556,364,1,E,Europe/Berlin,airport,OurAirports,Europe
9,580,453,450,903,Amsterdam Airport Schiphol,Amsterdam,Netherlands,AMS,EHAM,52.308601,4.763890,-11,1,E,Europe/Amsterdam,airport,OurAirports,Europe
12,1218,391,392,783,Barcelona International Airport,Barcelona,Spain,BCN,LEBL,41.297100,2.078460,12,1,E,Europe/Madrid,airport,OurAirports,Europe


#### АЗИЯ

In [45]:
asia_largest = route_count_merge[route_count_merge['Continent']=='Asia']\
.sort_values(by='Сумма взлет/посадок', ascending = False).head(5)
asia_largest

,Airport_ID,Arrivals,Destination,Сумма взлет/посадок,Name,City,Country,IATA,ICAO,Latitude,Longitude,Altitude,Timezone,DST,Tz database time zone,Type,Source,Continent
2,3364,535,534,1069,Beijing Capital International Airport,Beijing,China,PEK,ZBAA,40.080101,116.584999,116,8,U,Asia/Shanghai,airport,OurAirports,Asia
10,3406,411,414,825,Shanghai Pudong International Airport,Shanghai,China,PVG,ZSPD,31.143400,121.805000,13,8,U,Asia/Shanghai,airport,OurAirports,Asia
11,3316,408,412,820,Singapore Changi Airport,Singapore,Singapore,SIN,WSSS,1.350190,103.994003,22,8,N,Asia/Singapore,airport,OurAirports,Asia
13,3930,370,370,740,Incheon International Airport,Seoul,South Korea,ICN,RKSI,37.469101,126.450996,23,9,U,Asia/Seoul,airport,OurAirports,Asia
17,1701,358,361,719,Atatürk International Airport,Istanbul,Turkey,ISL,LTBA,40.976898,28.814600,163,3,E,Europe/Istanbul,airport,OurAirports,Asia


#### ЮЖНАЯ АМЕРИКА

In [46]:
south_america_largest = route_count_merge[route_count_merge['Continent']=='South America']\
.sort_values(by='Сумма взлет/посадок', ascending = False).head(5)
south_america_largest

,Airport_ID,Arrivals,Destination,Сумма взлет/посадок,Name,City,Country,IATA,ICAO,Latitude,Longitude,Altitude,Timezone,DST,Tz database time zone,Type,Source,Continent
74,2564,193,194,387,Guarulhos - Governador André Franco Montoro In...,Sao Paulo,Brazil,GRU,SBGR,-23.435556,-46.473057,2459,-3,S,America/Sao_Paulo,airport,OurAirports,South America
103,2709,152,155,307,El Dorado International Airport,Bogota,Colombia,BOG,SKBO,4.701590,-74.146900,8361,-5,U,America/Bogota,airport,OurAirports,South America
137,2789,115,115,230,Jorge Chávez International Airport,Lima,Peru,LIM,SPIM,-12.021900,-77.114305,113,-5,U,America/Lima,airport,OurAirports,South America
170,2531,98,99,197,Presidente Juscelino Kubistschek International...,Brasilia,Brazil,BSB,SBBR,-15.869167,-47.920834,3497,-3,S,America/Sao_Paulo,airport,OurAirports,South America
169,2560,98,96,194,Rio Galeão – Tom Jobim International Airport,Rio De Janeiro,Brazil,GIG,SBGL,-22.809999,-43.250557,28,-3,S,America/Sao_Paulo,airport,OurAirports,South America


#### СЕВЕРНАЯ АМЕРИКА

In [47]:
north_america_largest = route_count_merge[route_count_merge['Continent']=='North America']\
.sort_values(by='Сумма взлет/посадок', ascending = False).head(5)
north_america_largest

,Airport_ID,Arrivals,Destination,Сумма взлет/посадок,Name,City,Country,IATA,ICAO,Latitude,Longitude,Altitude,Timezone,DST,Tz database time zone,Type,Source,Continent
0,3682,915,911,1826,Hartsfield Jackson Atlanta International Airport,Atlanta,United States,ATL,KATL,33.636700,-84.428101,1026,-5,A,America/New_York,airport,OurAirports,North America
1,3830,558,550,1108,Chicago O'Hare International Airport,Chicago,United States,ORD,KORD,41.978600,-87.904800,672,-6,A,America/Chicago,airport,OurAirports,North America
6,3484,492,498,990,Los Angeles International Airport,Los Angeles,United States,LAX,KLAX,33.942501,-118.407997,125,-8,A,America/Los_Angeles,airport,OurAirports,North America
7,3670,469,467,936,Dallas Fort Worth International Airport,Dallas-Fort Worth,United States,DFW,KDFW,32.896801,-97.038002,607,-6,A,America/Chicago,airport,OurAirports,North America
8,3797,456,455,911,John F Kennedy International Airport,New York,United States,JFK,KJFK,40.639801,-73.778900,13,-5,A,America/New_York,airport,OurAirports,North America


#### АВСТРАЛИЯ

In [48]:
oceania_largest = route_count_merge[route_count_merge['Continent']=='Oceania']\
.sort_values(by='Сумма взлет/посадок', ascending = False).head(5)
oceania_largest

,Airport_ID,Arrivals,Destination,Сумма взлет/посадок,Name,City,Country,IATA,ICAO,Latitude,Longitude,Altitude,Timezone,DST,Tz database time zone,Type,Source,Continent
62,3361,208,202,410,Sydney Kingsford Smith International Airport,Sydney,Australia,SYD,YSSY,-33.946098,151.177002,21,10,O,Australia/Sydney,airport,OurAirports,Oceania
102,3320,152,144,296,Brisbane International Airport,Brisbane,Australia,BNE,YBBN,-27.384199,153.117004,13,10,N,Australia/Brisbane,airport,OurAirports,Oceania
112,3339,137,132,269,Melbourne International Airport,Melbourne,Australia,MEL,YMML,-37.673302,144.843002,434,10,O,Australia/Hobart,airport,OurAirports,Oceania
175,2006,96,117,213,Auckland International Airport,Auckland,New Zealand,AKL,NZAA,-37.008099,174.792007,23,12,Z,Pacific/Auckland,airport,OurAirports,Oceania
181,3351,92,90,182,Perth International Airport,Perth,Australia,PER,YPPH,-31.940300,115.967003,67,8,N,Australia/Perth,airport,OurAirports,Oceania


#### АНТАРКТИДА

In [49]:
antarctica_largest = route_count_merge[route_count_merge['Continent']=='Antarctica']\
.sort_values(by='Сумма взлет/посадок', ascending = False).head(5)
antarctica_largest

,Airport_ID,Arrivals,Destination,Сумма взлет/посадок,Name,City,Country,IATA,ICAO,Latitude,Longitude,Altitude,Timezone,DST,Tz database time zone,Type,Source,Continent


#### АФРИКА

In [50]:
africa_largest = route_count_merge[route_count_merge['Continent']=='Africa']\
.sort_values(by='Сумма взлет/посадок', ascending = False).head(5)
africa_largest

,Airport_ID,Arrivals,Destination,Сумма взлет/посадок,Name,City,Country,IATA,ICAO,Latitude,Longitude,Altitude,Timezone,DST,Tz database time zone,Type,Source,Continent
93,813,159,162,321,OR Tambo International Airport,Johannesburg,South Africa,JNB,FAOR,-26.139200,28.246000,5558,2,U,Africa/Johannesburg,airport,OurAirports,Africa
104,1128,151,152,303,Cairo International Airport,Cairo,Egypt,CAI,HECA,30.121901,31.405600,382,2,U,Africa/Cairo,airport,OurAirports,Africa
125,4059,130,128,258,Jomo Kenyatta International Airport,Nairobi,Kenya,NBO,HKJK,-1.319240,36.927799,5330,3,N,Africa/Nairobi,airport,OurAirports,Africa
128,1074,125,125,250,Mohammed V International Airport,Casablanca,Morocco,CMN,GMMN,33.367500,-7.589970,656,0,N,Africa/Casablanca,airport,OurAirports,Africa
156,1107,105,109,214,Addis Ababa Bole International Airport,Addis Ababa,Ethiopia,ADD,HAAB,8.977890,38.799301,7630,3,U,Africa/Addis_Ababa,airport,OurAirports,Africa


#### Теперь обобъединим все датафреймы, чтобы потом нанести их на карту

In [51]:
biggest_po_continent = pd.concat([eur_largest, asia_largest, 
                                  north_america_largest, south_america_largest, 
                                  oceania_largest, africa_largest, antarctica_largest])
biggest_po_continent

,Airport_ID,Arrivals,Destination,Сумма взлет/посадок,Name,City,Country,IATA,ICAO,Latitude,Longitude,Altitude,Timezone,DST,Tz database time zone,Type,Source,Continent
3,507,527,524,1051,London Heathrow Airport,London,United Kingdom,LHR,EGLL,51.470600,-0.461941,83,0,E,Europe/London,airport,OurAirports,Europe
4,1382,524,517,1041,Charles de Gaulle International Airport,Paris,France,CDG,LFPG,49.012798,2.550000,392,1,E,Europe/Paris,airport,OurAirports,Europe
5,340,497,493,990,Frankfurt am Main Airport,Frankfurt,Germany,FRA,EDDF,50.033333,8.570556,364,1,E,Europe/Berlin,airport,OurAirports,Europe
9,580,453,450,903,Amsterdam Airport Schiphol,Amsterdam,Netherlands,AMS,EHAM,52.308601,4.763890,-11,1,E,Europe/Amsterdam,airport,OurAirports,Europe
12,1218,391,392,783,Barcelona International Airport,Barcelona,Spain,BCN,LEBL,41.297100,2.078460,12,1,E,Europe/Madrid,airport,OurAirports,Europe
2,3364,535,534,1069,Beijing Capital International Airport,Beijing,China,PEK,ZBAA,40.080101,116.584999,116,8,U,Asia/Shanghai,airport,OurAirports,Asia
10,3406,411,414,825,Shanghai Pudong International Airport,Shanghai,China,PVG,ZSPD,31.143400,121.805000,13,8,U,Asia/Shanghai,airport,OurAirports,Asia
11,3316,408,412,820,Singapore Changi Airport,Singapore,Singapore,SIN,WSSS,1.350190,103.994003,22,8,N,Asia/Singapore,airport,OurAirports,Asia
13,3930,370,370,740,Incheon International Airport,Seoul,South Korea,ICN,RKSI,37.469101,126.450996,23,9,U,Asia/Seoul,airport,OurAirports,Asia
17,1701,358,361,719,Atatürk International Airport,Istanbul,Turkey,ISL,LTBA,40.976898,28.814600,163,3,E,Europe/Istanbul,airport,OurAirports,Asia


#### Для нанесения точек на карту используем ту же библиотеку Folium

In [52]:
m_biggest = folium.Map()

#Через цикл добавляем все точки, как делали ранее

lat = biggest_po_continent['Latitude']
lon = biggest_po_continent['Longitude']
iata = biggest_po_continent['IATA']

for lat, lon, iata in zip(lat , lon, iata) :
    folium.Marker(location=[lat, lon], popup='IATA: '+str(iata), icon=folium.Icon(color = 'green')).add_to(m_biggest)
m_biggest

#### Получим IATA-коды крупнейших аэропортов

In [53]:
top_iata = set(biggest_po_continent['IATA'].dropna())

#### Отфильтруем маршруты, где оба конца входят в топ

In [54]:
double_routes = routes[(routes["Source Airport"].isin(top_iata)) & 
(routes["Destination Airport"].isin(top_iata))]

route_set = set(zip(double_routes["Source Airport"],
    double_routes["Destination Airport"]))

#### Далее создаем цикл, который проходится по маршрутам и распределяет их

In [55]:
bidirectional_routes = set() # Это множество для двусторонних маршрутов
unidirectional_routes = set() # Это множество для односторонних маршрутов

for route in route_set:
    reversed_route = (route[1], route[0]) # Это обратный маршрут для проверки
    if reversed_route in route_set: # Тут уже происходит сама проверка: то есть входит ли обратный маршрут в созданное ранее множество
        bidirectional_routes.add(tuple(sorted(route)))  # сохраняем упорядоченно, чтобы не дублировалось
    else: # Ну, соответственно, если проверка не проходит, значит, это односторонний маршрут
        unidirectional_routes.add(route)

In [56]:
unidirectional_routes

{('DFW', 'BNE'), ('FRA', 'NBO'), ('PEK', 'JNB'), ('SYD', 'DFW')}

#### Вот мы и определили одностороннюю и двустороннюю связи. Теперь с помощью библиотеки Folium нанесем линии на карту.

In [57]:
for source, destination in bidirectional_routes: # Наносим красные линии - двусторонние связи
    source_info = biggest_po_continent[biggest_po_continent['IATA'] == source].iloc[0]
    destination_info = biggest_po_continent[biggest_po_continent['IATA'] == destination].iloc[0]
    
    source_coords = [source_info['Latitude'], source_info['Longitude']] # Координаты вылета
    dest_coords = [destination_info['Latitude'], destination_info['Longitude']] # Координаты прилета
    
    folium.PolyLine(
        locations=[source_coords, dest_coords],
        color='red',
        weight=2
    ).add_to(m_biggest)
m_biggest

In [58]:
# Добавляем односторонние маршруты тем же способом
for source, destination in unidirectional_routes:
    source_info = biggest_po_continent[biggest_po_continent['IATA'] == source].iloc[0]
    destination_info = biggest_po_continent[biggest_po_continent['IATA'] == destination].iloc[0]
    
    source_coords = [source_info['Latitude'], source_info['Longitude']]
    dest_coords = [destination_info['Latitude'], destination_info['Longitude']]
    
    folium.PolyLine(
        locations=[source_coords, dest_coords],
        color='green',
        weight=2
    ).add_to(m_biggest)

# Отображаем карту
m_biggest